<a href="https://colab.research.google.com/github/jaloaiza/genaiassignments/blob/main/assignments/07/FinalProjDatasetGen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Generate Synthetic Training Data

<a target="_blank" href="https://colab.research.google.com/github/simonguest/CS-394/blob/main/src/06/notebooks/generate-synthetic.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
<a target="_blank" href="https://github.com/simonguest/CS-394/raw/refs/heads/main/src/06/notebooks/generate-synthetic.ipynb">
  <img src="https://img.shields.io/badge/Download_.ipynb-blue" alt="Download .ipynb"/>
</a>

## Data generation settings

In [1]:
NUM_TRAIN_EXAMPLES = 500  # @param {type:"number"}
NUM_VAL_EXAMPLES = 100  # @param {type:"number"}
NUM_TEST_EXAMPLES = 50 # @param {type:"number"}
TEMPERATURE = 0.8  # @param {type:"number"}
BATCH_SIZE = 5  # @param {type:"number"}

DATA_FOLDER = "./data/generated"
!mkdir -p {DATA_FOLDER}

DATAGEN_URL = "https://openrouter.ai/api/v1"
DATAGEN_MODEL = "openai/gpt-5-nano"

## Dataset diversity

In [2]:
COMMUNICATION_STYLES = [
    "casual slang",
    "neutral",
    "formal",
    "terse",
    "expressive",
    "sarcastic",
    "confused",
    "curious",
    "impatient",
    "friendly"
]

PLAYER_TYPES = [
    "explorer",
    "competitor",
    "casual player",
    "roleplayer",
    "min maxer",
    "social player",
    "troll",
    "helper"
]

WRITING_PATTERNS = [
    "clean",
    "no caps",
    "minor typos",
    "heavy typos",
    "shortened words",
    "messy spacing",
    "run on"
]

VERBOSITY = [
    "one word",
    "one sentence",
    "two sentences"
]

COMMUNICATION_STYLE_WEIGHTS = [0.3, 0.2, 0.1, 0.1, 0.1, 0.05, 0.05, 0.05, 0.025, 0.025]

PLAYER_TYPE_WEIGHTS = [0.2, 0.15, 0.25, 0.1, 0.1, 0.1, 0.05, 0.05]

WRITING_PATTERN_WEIGHTS = [0.3, 0.2, 0.2, 0.1, 0.1, 0.05, 0.05]

VERBOSITY_WEIGHTS = [0.2, 0.5, 0.3]


## Model for structured output

In [3]:
from pydantic import BaseModel

class ProfilerExample(BaseModel):
    message: str

class ProfilerBatch(BaseModel):
    examples: list[ProfilerExample]

## Get OpenRouter API key

In [4]:
import sys
import os
from dotenv import load_dotenv

if 'google.colab' in sys.modules:
  from google.colab import userdata # type:ignore
  os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
else:
  load_dotenv()

## Conversation generation functions

In [5]:
import openai
import os
from dataclasses import dataclass

client = openai.OpenAI(
    base_url=DATAGEN_URL,
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

def generate_completion(prompt: str) -> ProfilerExample | None:
    response = client.responses.parse(
        model=DATAGEN_MODEL,
        input=[{"role": "user", "content": prompt}],
        temperature=TEMPERATURE,
        stream=False,
        text_format=ProfilerExample
    )

    return response.output_parsed

def create_conversation(
    communication_style: str,
    player_type: str,
    writing_pattern: str,
    verbosity: str
) -> ProfilerExample | None:

    prompt = f"""
    You are generating realistic user chat messages to train a video game chatbot.

    Generate a single message ({verbosity}) that fits this profile:

    - Communication style: {communication_style}
    - Player type: {player_type}
    - Writing pattern: {writing_pattern}
    """

    return generate_completion(prompt)

def create_conversation_batch(profile: dict, count: int) -> ProfilerBatch | None:
    prompt = f"""
    You are generating realistic user chat messages to train a video game chatbot.

    Generate exactly {count} unique messages that all fit this profile:

    - Communication style: {profile['communication_style']}
    - Player type: {profile['player_type']}
    - Writing pattern: {profile['writing_pattern']}
    - Verbosity: {profile['verbosity']}
    """

    response = client.responses.parse(
        model=DATAGEN_MODEL,
        input=[{"role": "user", "content": prompt}],
        temperature=TEMPERATURE,
        stream=False,
        text_format=ProfilerBatch
    )

    return response.output_parsed

## Dataset generation functions

In [6]:
import random
import json
from tqdm import tqdm

def generate_dataset(num_examples: int, filename: str) -> None:
    with open(filename, "w", encoding="utf-8") as f:
        generated = 0
        pbar = tqdm(total=num_examples)
        while generated < num_examples:
            count = min(BATCH_SIZE, num_examples - generated)
            profile = {
                "communication_style": random.choice(COMMUNICATION_STYLES),
                "player_type": random.choice(PLAYER_TYPES),
                "writing_pattern": random.choice(WRITING_PATTERNS),
                "verbosity": random.choices(VERBOSITY, weights=VERBOSITY_WEIGHTS)[0],
            }

            batch_result = None
            while batch_result is None:
                batch_result = create_conversation_batch(profile, count)

            for example in batch_result.examples:
                template = {
                    "message": example.message,
                    "labels": {
                        "communication_style": profile["communication_style"],
                        "player_type": profile["player_type"],
                        "writing_pattern": profile["writing_pattern"],
                        "verbosity": profile["verbosity"]
                    }
                }
                f.write(json.dumps(template, ensure_ascii=False) + "\n")

            generated += count
            pbar.update(count)
            f.flush()

        pbar.close()

## Generate all the data!

In [ ]:
from datetime import datetime

TRAIN_FILE = f"{DATA_FOLDER}/train_{datetime.now().strftime('%Y-%m-%d-%H:%M:%S')}.jsonl"
VALID_FILE = f"{DATA_FOLDER}/valid_{datetime.now().strftime('%Y-%m-%d-%H:%M:%S')}.jsonl"
TEST_FILE = f"{DATA_FOLDER}/test_{datetime.now().strftime('%Y-%m-%d-%H:%M:%S')}.jsonl"

generate_dataset(NUM_TRAIN_EXAMPLES, TRAIN_FILE)
generate_dataset(NUM_VAL_EXAMPLES, VALID_FILE)
generate_dataset(NUM_TEST_EXAMPLES, TEST_FILE)


  1%|          | 5/500 [00:24<40:20,  4.89s/it]